# XGBoost–SARIMA inference
Inference only: download the `champion` raw-input hybrid pipeline from W&B Model Registry, predict `test.csv`, and create a Kaggle-ready submission.

In [ ]:
%pip install -q "xgboost>=3,<4" "statsmodels>=0.14,<1" "wandb>=0.19,<1" "cloudpickle>=3,<4"

In [ ]:
from pathlib import Path
import warnings, cloudpickle, numpy as np, pandas as pd, wandb
warnings.filterwarnings('ignore')
DATA_DIR=Path('/content/drive/MyDrive/walmart_competition_data') if Path('/content').exists() else Path('../../data')
OUTPUT_DIR = Path('/content/drive/MyDrive/walmart_competition_inference/xgboost_sarima') if Path('/content').exists() else Path('outputs')
MODEL_DOWNLOAD_DIR = Path('/content/artifacts/wandb_registry_xgboost_sarima')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True); MODEL_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
WANDB_PROJECT = 'Walmart-Recruiting---Store-Sales-Forecasting'
REGISTRY_ARTIFACT_URI = 'wandb-registry-model/Walmart_XGBoost_SARIMA_Pipeline:champion'
raw = pd.read_csv(DATA_DIR/'test.csv', parse_dates=['Date'])
assert raw.columns.tolist() == ['Store','Dept','Date','IsHoliday']
assert not raw.duplicated(['Store','Dept','Date']).any()

## Download and load the champion pipeline

In [ ]:
run = wandb.init(project=WANDB_PROJECT, name='XGBoost_SARIMA_Registry_Inference', job_type='inference', reinit=True)
artifact = run.use_artifact(REGISTRY_ARTIFACT_URI, type='model')
artifact_dir = Path(artifact.download(root=MODEL_DOWNLOAD_DIR))
pipeline_files = list(artifact_dir.rglob('*.pkl'))
assert len(pipeline_files) == 1, f'Expected one pipeline, found: {pipeline_files}'
with pipeline_files[0].open('rb') as file: pipeline = cloudpickle.load(file)
assert hasattr(pipeline, 'predict'); print(getattr(pipeline, 'config', {}))

## Predict directly from raw test rows

In [ ]:
predictions = np.asarray(pipeline.predict(raw), dtype=float)
assert predictions.shape == (len(raw),) and np.isfinite(predictions).all()
submission = pd.DataFrame({'Id': raw.Store.astype(str)+'_'+raw.Dept.astype(str)+'_'+raw.Date.dt.strftime('%Y-%m-%d'), 'Weekly_Sales': predictions})
submission_path = OUTPUT_DIR/'xgboost_sarima_registry_submission.csv'
submission.to_csv(submission_path, index=False)
submission_artifact = wandb.Artifact('xgboost-sarima-registry-submission', type='submission', metadata={'registry_artifact':REGISTRY_ARTIFACT_URI})
submission_artifact.add_file(str(submission_path)); run.log_artifact(submission_artifact); run.finish()
print(submission_path); display(submission.head()); display(submission.Weekly_Sales.describe())